<center>
  <img src="https://upload.wikimedia.org/wikipedia/commons/6/68/Logo_FPT_Education.png" width="300" alt="FPT Education logo" />
</center>

# II. DATA COLLECTION, EXPLORATION, AND PREPROCESSING

**Dự án:** ADY201m - AI, Data Science with Python & SQL  
**Bộ dữ liệu:** Brazilian E-Commerce Public Dataset by Olist

**Phạm vi:** dự đoán `low_review` từ hai biến đầu vào `delay_days` và `item_count`.

---

## 2.1. Data Collection

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
RAW_DIR = next(p / "data" / "raw" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
OUT_DIR = RAW_DIR.parent / "processed"
print(RAW_DIR)

D:\FPT UNIVERSITY\Kì 3\ADY201m\ADY291c-AI2107-Group2\data\raw


In [3]:
orders = pd.read_csv(RAW_DIR / "olist_orders_dataset.csv")
order_items = pd.read_csv(RAW_DIR / "olist_order_items_dataset.csv")
reviews = pd.read_csv(RAW_DIR / "olist_order_reviews_dataset.csv")

In [4]:
orders = orders[["order_id", "order_status", "order_delivered_customer_date", "order_estimated_delivery_date"]]
order_items = order_items[["order_id", "order_item_id"]]
reviews = reviews[["order_id", "review_id", "review_score"]]

In [5]:
for df in [orders, order_items, reviews]:
    print(df.shape)
    display(df.head(3))

(99441, 4)


,order_id,order_status,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08-17 18:06:29,2018-09-04 00:00:00


(112650, 2)


,order_id,order_item_id
0,00010242fe8c5a6d1ba2dd792cb16214,1
1,00018f77f2f0320c557190d7a144bdd3,1
2,000229ec398224ef6ca0657da4fc703e,1


(99224, 3)


,order_id,review_id,review_score
0,73fc7af87114b39712e6da79b0a377eb,7bc2406110b926393aa56f80a40eba40,4
1,a548910a1c6147796b98fdf73dbeba33,80e641a11e56f04c1ad469d5645fdfde,5
2,f9e4b658b201a9f2ecdecbb34bed034b,228ce5500dc1d8e020d8d1322874b6f0,5


**Nhận xét:** đã đọc đủ 3 bảng và chỉ giữ 9 cột phục vụ bài toán.

## 2.2. Data Understanding

In [6]:
orders.info()
order_items.info()
reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 4 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   order_status                   99441 non-null  str  
 2   order_delivered_customer_date  96476 non-null  str  
 3   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(4)
memory usage: 10.5 MB
<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 2 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   order_id       112650 non-null  str  
 1   order_item_id  112650 non-null  int64
dtypes: int64(1), str(1)
memory usage: 5.2 MB
<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   order_id      992

In [7]:
print(orders.isnull().sum())
print(order_items.isnull().sum())
print(reviews.isnull().sum())

order_id                            0
order_status                        0
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
order_id         0
order_item_id    0
dtype: int64
order_id        0
review_id       0
review_score    0
dtype: int64


In [8]:
print("Dòng trùng toàn bộ - orders:", orders.duplicated().sum())
print("Dòng trùng toàn bộ - order_items:", order_items.duplicated().sum())
print("Dòng trùng toàn bộ - reviews:", reviews.duplicated().sum())

Dòng trùng toàn bộ - orders: 0
Dòng trùng toàn bộ - order_items: 0
Dòng trùng toàn bộ - reviews: 0


### Kiểm tra khóa và trùng lặp

In [9]:
print("Trùng order_id (orders):", orders["order_id"].duplicated().sum())
print("Trùng (order_id, order_item_id):", order_items.duplicated(["order_id", "order_item_id"]).sum())
print("Trùng review_id (reviews):", reviews["review_id"].duplicated().sum())
print("Order có nhiều hơn 1 review:", (reviews["order_id"].value_counts() > 1).sum())
print("Số review tối đa trên 1 order:", reviews["order_id"].value_counts().max())

Trùng order_id (orders): 0
Trùng (order_id, order_item_id): 0
Trùng review_id (reviews): 814


Order có nhiều hơn 1 review: 547


Số review tối đa trên 1 order: 3


In [10]:
print(orders["order_status"].value_counts())
print(reviews["review_score"].value_counts().sort_index())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64


In [11]:
orders[["order_delivered_customer_date", "order_estimated_delivery_date"]].dtypes

order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

**Nhận xét:**
- `orders`: `order_id` không trùng, nhưng 2.965 dòng thiếu ngày giao thực tế.
- `order_items`: khóa `(order_id, order_item_id)` duy nhất, không có dòng trùng.
- `reviews`: 814 `review_id` xuất hiện ở nhiều đơn khác nhau nhưng cùng điểm số, nên không xóa theo `review_id`; 547 đơn có hơn 1 review (tối đa 3).
- `review_score` chỉ nhận 1-5, không thiếu giá trị.
- Hai cột ngày đang ở kiểu chuỗi, cần chuyển sang `datetime`.

## 2.3. Data Preprocessing

### 1. Xử lý bản ghi trùng theo khóa

In [12]:
orders = orders.drop_duplicates(subset="order_id")
order_items = order_items.drop_duplicates(subset=["order_id", "order_item_id"])
reviews = reviews.drop_duplicates()

print(orders.shape, order_items.shape, reviews.shape)

(99441, 4) (112650, 2) (99224, 3)


### 2. Chuyển cột ngày sang datetime

In [13]:
date_cols = ["order_delivered_customer_date", "order_estimated_delivery_date"]
orders[date_cols] = orders[date_cols].apply(pd.to_datetime)
orders[date_cols].dtypes

order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

### 3. Chỉ giữ đơn đã giao

In [14]:
orders = orders[orders["order_status"] == "delivered"]
print(orders.shape)

(96478, 4)


### 4. Loại đơn thiếu ngày giao thực tế hoặc dự kiến

In [15]:
orders = orders.dropna(subset=date_cols)
print(orders.shape)

(96470, 4)


### 5. Kiểm tra review_score hợp lệ

In [16]:
print("Toàn bộ trong khoảng 1-5:", reviews["review_score"].between(1, 5).all())
reviews = reviews[reviews["review_score"].between(1, 5)]
print(reviews.shape)

Toàn bộ trong khoảng 1-5: True
(99224, 3)


### 6. Xử lý nhiều review trên một order_id

Giữ bản ghi đánh giá cuối cùng của mỗi `order_id` trong file (lần cập nhật sau cùng), mỗi đơn còn đúng 1 review.

In [17]:
reviews = reviews.drop_duplicates(subset="order_id", keep="last")
print(reviews.shape, "| Trùng order_id:", reviews["order_id"].duplicated().sum())

(98673, 3) | Trùng order_id: 0


### 7. Tổng hợp order_items theo order_id

In [18]:
item_count = order_items.groupby("order_id")["order_item_id"].count().reset_index(name="item_count")
print(item_count.shape)
item_count.head()

(98666, 2)


,order_id,item_count
0,00010242fe8c5a6d1ba2dd792cb16214,1
1,00018f77f2f0320c557190d7a144bdd3,1
2,000229ec398224ef6ca0657da4fc703e,1
3,00024acbcdf0a6daa1e931b038114c75,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1


### 8. Merge 3 bảng theo order_id

In [19]:
df = orders.merge(item_count, on="order_id", how="inner").merge(reviews, on="order_id", how="inner")
print(orders.shape, "->", df.shape)
print("Trùng order_id sau merge:", df["order_id"].duplicated().sum())

(96470, 4) -> (95824, 7)
Trùng order_id sau merge: 0


**Nhận xét:** tổng hợp `order_items` và xử lý review trước khi merge nên quan hệ là 1-1, số dòng không bị nhân bản.

## 2.4. Feature Engineering

### 1. delay_days - số ngày giao thực tế lệch so với dự kiến

In [20]:
df["delay_days"] = (df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]).dt.days

### 2. low_review - biến mục tiêu

In [21]:
df["low_review"] = np.where(df["review_score"] <= 2, 1, 0)

### 3. Dataset cuối cùng

Tạo hai bộ dữ liệu từ cùng một nguồn đã xử lý:
- `olist_model_dataset.csv`: dùng cho Machine Learning, gồm `order_id`, `delay_days`, `item_count`, `low_review`.
- `olist_analysis_dataset.csv`: dùng cho trực quan hóa, giữ thêm `review_score` để vẽ biểu đồ theo điểm đánh giá.

In [22]:
analysis_df = df[["order_id", "delay_days", "item_count", "review_score", "low_review"]]
model_df = analysis_df[["order_id", "delay_days", "item_count", "low_review"]]

print(analysis_df.shape, model_df.shape)
model_df.head()

(95824, 5) (95824, 4)


,order_id,delay_days,item_count,low_review
0,e481f51cbdc54678b7cc49136f2d6af7,-8,1,0
1,53cdb2fc8bc7dce0b6741e2150273451,-6,1,0
2,47770eb9100c2d0c44946d9cf07ec65d,-18,1,0
3,949d5b44dbf5de918fe9c16f97b45f8a,-13,1,0
4,ad21c59c0840e6cb83a9ceb5573f8159,-10,1,0


In [23]:
model_df.info()
print(model_df.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 95824 entries, 0 to 95823
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   order_id    95824 non-null  str  
 1   delay_days  95824 non-null  int64
 2   item_count  95824 non-null  int64
 3   low_review  95824 non-null  int64
dtypes: int64(3), str(1)
memory usage: 5.9 MB
order_id      0
delay_days    0
item_count    0
low_review    0
dtype: int64


In [24]:
print("Trùng order_id:", model_df["order_id"].duplicated().sum())
print(model_df["low_review"].value_counts().sort_index())
print((model_df["low_review"].value_counts(normalize=True) * 100).round(2))

Trùng order_id: 0
low_review
0    83556
1    12268
Name: count, dtype: int64
low_review
0    87.2
1    12.8
Name: proportion, dtype: float64


In [25]:
OUT_DIR.mkdir(exist_ok=True)
model_df.to_csv(OUT_DIR / "olist_model_dataset.csv", index=False)
analysis_df.to_csv(OUT_DIR / "olist_analysis_dataset.csv", index=False)

print("Đã lưu:", OUT_DIR / "olist_model_dataset.csv")
print("Đã lưu:", OUT_DIR / "olist_analysis_dataset.csv")

Đã lưu: D:\FPT UNIVERSITY\Kì 3\ADY201m\ADY291c-AI2107-Group2\data\processed\olist_model_dataset.csv
Đã lưu: D:\FPT UNIVERSITY\Kì 3\ADY201m\ADY291c-AI2107-Group2\data\processed\olist_analysis_dataset.csv


**Kết luận:** hai bộ dữ liệu đều có một dòng cho mỗi `order_id` và không thiếu giá trị. `olist_model_dataset.csv` dùng cho Machine Learning, còn `olist_analysis_dataset.csv` được dùng cho phần trực quan hóa vì có thêm `review_score`.